In [ ]:
!pip install ultralytics opencv-python matplotlib labelImg

In [ ]:
import os

image_path = "data"
print(len(os.listdir(image_path)))

In [ ]:
import os

# Create folder structure
os.makedirs("dataset/images/train", exist_ok=True)
os.makedirs("dataset/images/val", exist_ok=True)
os.makedirs("dataset/labels/train", exist_ok=True)
os.makedirs("dataset/labels/val", exist_ok=True)

print("Folders created successfully")

In [ ]:
import xml.etree.ElementTree as ET
import os

xml_path = "data/annotations.xml"
output_dir = "labels_temp"

os.makedirs(output_dir, exist_ok=True)

class_map = {
    "spalling": 0,
    "honeycombing": 1,
    "rebar": 2,
    "damage": 3
}

tree = ET.parse(xml_path)
root = tree.getroot()

for image in root.findall("image"):
    filename = image.get("name")
    width = float(image.get("width"))
    height = float(image.get("height"))

    txt_filename = filename.replace(".JPG", ".txt")
    txt_path = os.path.join(output_dir, txt_filename)

    with open(txt_path, "w") as f:
        for poly in image.findall("polygon"):
            label = poly.get("label")

            if label not in class_map:
                continue

            class_id = class_map[label]

            points = poly.get("points").split(";")

            yolo_points = []

            for p in points:
                x, y = map(float, p.split(","))

                # Normalize
                x = x / width
                y = y / height

                yolo_points.append(f"{x} {y}")

            line = f"{class_id} " + " ".join(yolo_points)
            f.write(line + "\n")

print("✅ Segmentation labels created!")

In [ ]:
with open("labels_temp/MAX_0441.txt", "r") as f:
    print(f.read())

In [ ]:
import os
import random
import shutil

# Get images
images = [f for f in os.listdir("data") if f.endswith(".JPG")]

# Shuffle
random.shuffle(images)

# 80% train, 20% val
split = int(0.8 * len(images))

train_imgs = images[:split]
val_imgs = images[split:]

def copy_files(img_list, split):
    for img in img_list:
        label = img.replace(".JPG", ".txt")
        
        label_path = f"labels/{label}"
        
        # Skip if label not found
        if not os.path.exists(label_path):
            continue
        
        shutil.copy(f"data/{img}", f"dataset/images/{split}/{img}")
        shutil.copy(label_path, f"dataset/labels/{split}/{label}")
        
copy_files(train_imgs, "train")
copy_files(val_imgs, "val")

print("Train/Val split done ✅")

In [ ]:
import shutil
import os

# Remove wrong empty labels folder (optional but clean)
shutil.rmtree("dataset/labels", ignore_errors=True)

# Copy correct labels
shutil.copytree("labels_temp", "dataset/labels")

print("Labels moved successfully ✅")

In [ ]:
import os

print("Labels inside dataset:", len(os.listdir("dataset/labels")))
print(os.listdir("dataset/labels")[:5])

In [ ]:
import shutil

shutil.rmtree("dataset/images", ignore_errors=True)
shutil.rmtree("dataset/labels_split", ignore_errors=True)

In [ ]:
import os

os.makedirs("dataset/images/train", exist_ok=True)
os.makedirs("dataset/images/val", exist_ok=True)

os.makedirs("dataset/labels_split/train", exist_ok=True)
os.makedirs("dataset/labels_split/val", exist_ok=True)

In [ ]:
import os
import random
import shutil

images = [f for f in os.listdir("data") if f.endswith(".JPG")]
random.shuffle(images)

split = int(0.8 * len(images))

train_imgs = images[:split]
val_imgs = images[split:]

def copy_files(img_list, split_type):
    for img in img_list:
        label = img.replace(".JPG", ".txt")
        
        shutil.copy(f"data/{img}", f"dataset/images/{split_type}/{img}")
        shutil.copy(f"dataset/labels/{label}", f"dataset/labels_split/{split_type}/{label}")

copy_files(train_imgs, "train")
copy_files(val_imgs, "val")

print("Split done ✅")

In [ ]:
import os

print("Train images:", len(os.listdir("dataset/images/train")))
print("Train labels:", len(os.listdir("dataset/labels_split/train")))

print("Val images:", len(os.listdir("dataset/images/val")))
print("Val labels:", len(os.listdir("dataset/labels_split/val")))

In [ ]:
data_yaml = """
path: dataset

train: images/train
val: images/val

names:
  0: spalling
  1: honeycombing
  2: rebar
  3: damage
"""

with open("dataset.yaml", "w") as f:
    f.write(data_yaml)

print("dataset.yaml created ✅")

In [ ]:
with open("dataset.yaml", "r") as f:
    print(f.read())

In [ ]:
import shutil

shutil.rmtree("dataset/labels", ignore_errors=True)
shutil.move("dataset/labels_split", "dataset/labels")

print("Labels fixed ✅")

In [ ]:
!pip install ultralytics

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open("runs/detect/train/results.png")

plt.figure(figsize=(12,6))
plt.imshow(img)
plt.axis("off")
plt.show()

In [ ]:
from IPython.display import display
from PIL import Image

img = Image.open("runs/detect/train/results.png")
display(img)

In [ ]:
from ultralytics import YOLO

# Load your trained model
model = YOLO("runs/detect/train/weights/best.pt")

# Run prediction
results = model.predict(
    source="data",   # your images folder
    save=True,
    imgsz=640,
    conf=0.25
)

In [ ]:


img = Image.open("runs/detect/predict/" + os.listdir("runs/detect/predict")[1])
display(img)

In [ ]:
from ultralytics import YOLO

# 1. Load your best trained model
model = YOLO("runs/detect/train/weights/best.pt")

# 2. Run validation to get detailed metrics
metrics = model.val()

# 3. Print the specific results
print(f"Mean Average Precision (mAP@50): {metrics.seg.map50}") # For segmentation
print(f"Precision: {metrics.seg.p}") # Precision per class
print(f"Recall: {metrics.seg.r}")    # Recall per class
print(f"F1-Score: {metrics.seg.f1}")  # F1-Score per class

In [ ]:
from ultralytics import YOLO

# 1. Load the Segmentation-specific model
# This is a NEW model instance. It will not conflict with your 'model' from previous cells.
model_seg = YOLO("yolov8n-seg.pt") 

# 2. Start the new training
# This will create a NEW folder in 'runs/segment/train'
model_seg.train(
    data="dataset.yaml",
    epochs=50,
    imgsz=640,
    batch=4,       # You can try 4; if it crashes, change back to 2
    device="cpu",  # Keep as cpu if you don't have a GPU
    task="segment" # This ensures it looks for polygons, not just boxes
)

In [ ]:
from ultralytics import YOLO

# 1. Load the segmentation model you just trained
# Path might be runs/segment/train2 if you ran training multiple times
model = YOLO("runs/segment/train/weights/best.pt")

# 2. Run prediction on your specific test image
results = model.predict(
    source="dataset/images/test/", 
    save=True,        # Saves the visual result (masks overlaid)
    save_txt=True,    # Saves the coordinates of the masks
    conf=0.25,        # Adjust this to filter out weak detections
    project="test_results",
    name="single_image"
)

# 3. Display the result directly in the notebook
for r in results:
    # Set line_width to 1 or 2 for smaller text and thinner boxes
    im_array = r.plot(line_width=7, font_size=15)  
    
    from PIL import Image
    # Convert BGR (OpenCV format) to RGB for PIL
    display(Image.fromarray(im_array[..., ::-1]))

In [40]:
# Validate the model
metrics = model.val()

# Access segmentation-specific metrics
print("--- Segmentation Metrics ---")
print(f"mAP@50: {metrics.seg.map50:.4f}")   # Mean Average Precision
print(f"Precision: {metrics.seg.mp:.4f}")    # Mean Precision
print(f"Recall: {metrics.seg.mr:.4f}")       # Mean Recall
print(f"F1-Score: {metrics.seg.f1.mean():.4f}")

# Print class-wise performance
for i, name in metrics.names.items():
    print(f"Class {name}: Precision={metrics.seg.p[i]:.3f}, Recall={metrics.seg.r[i]:.3f}")

Ultralytics 8.4.30  Python-3.13.9 torch-2.11.0+cpu CPU (11th Gen Intel Core i7-11700K @ 3.60GHz)
val: Fast image access  (ping: 0.00.0 ms, read: 2625.6118.8 MB/s, size: 18717.0 KB)
val: Scanning C:\Users\Pdllab\chimneyf\dataset\labels\val.cache... 96 images, 37 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 96/96 40.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 4.3s/it 25.9s5.3ss
                   all         96        205      0.774       0.41      0.502      0.274      0.783      0.415      0.494      0.192
              spalling         21         25      0.868      0.264      0.424      0.224      0.868      0.265      0.418     0.0794
          honeycombing         22         51      0.696      0.471      0.526      0.264      0.742       0.51      0.531      0.224
                 rebar         36         84      0.731      0.262      0.293      0.159      0.795  